# ECE 214B Project 2: Submission-Ready Colab Notebook

This is the clean submission runner for the dementia speech detection project. It is designed to be opened in Google Colab from the packaged Drive folder, rerun the required final pipeline, expose every additional experiment tried, and export compact result files for submission.

All modeling logic lives in `project/scripts/` and `project/src/`. This notebook intentionally contains no raw participant data, no API keys, and no pasted protected transcript content.

## What This Notebook Covers

Default-run sections reproduce the required and final defensible session-level results: dataset audit, text/acoustic baselines, transformer baselines, multimodal fusion, clinical thresholding, marker analysis, final scene-fusion analysis, statistical testing, metadata validation, and result collection.

Optional sections cover expensive or exploratory experiments that were tried: ASR-error features, LLM clinical markers, Cookie Theft scene graph/event triples, hard-case adjudication, multiview profiles, alternative fusion/triage designs, semantic/disfluency analyses, subject aggregation, and related ablations. These are controlled by flags so a grader can see the complete experiment inventory without accidentally spending API budget or rerunning long searches.

## Setup

Edit `DROP_DIR` and `DATA_ZIP` only. Keep the dataset zip separate in Google Drive. If running any LLM section, set `OPENAI_API_KEY` in the Colab runtime environment or Colab secrets before executing validation.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

DROP_DIR = "/content/drive/MyDrive/214B_colab_run_final_final"
DATA_ZIP = "/content/drive/MyDrive/path/to/S26_ECE_214B_Mini_Project_2.zip"
EXTRACT_DIR = "/content/214B_data"

# Required/final run controls. These are suitable for submission reruns.
RUN_REQUIRED_PIPELINE = True
RUN_FINAL_ANALYSES = True
RUN_STATISTICS_AND_VALIDATION = True

# Expensive or API-dependent experiment controls. Leave false unless intentionally rerunning them.
RUN_ASR_EXPERIMENTS = False
RUN_LLM_CLINICAL_MARKERS = False
RUN_LLM_SCENE_GRAPH_BATCHES = False
RUN_LLM_EVENT_TRIPLE_BATCHES = False
RUN_LLM_EXTRA_EXPLORATIONS = False
RUN_EXPLORATORY_NO_API_EXPERIMENTS = False
RUN_HUBERT_LAYER_SWEEP = False

LLM_MODEL = "gpt-4o-mini"
LLM_SCENE_GRAPH_BATCH_SIZE = 50
LLM_EVENT_TRIPLE_BATCH_SIZE = 50

PROJECT_DIR = f"{DROP_DIR}/project"
OUTPUT_ZIP = f"{DROP_DIR}/outputs/submission_outputs.zip"
SUMMARY_ZIP = f"{DROP_DIR}/outputs/submission_summary_files.zip"

print("DROP_DIR:", DROP_DIR)
print("DATA_ZIP:", DATA_ZIP)
print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_REQUIRED_PIPELINE:", RUN_REQUIRED_PIPELINE)
print("RUN_FINAL_ANALYSES:", RUN_FINAL_ANALYSES)
print("RUN_STATISTICS_AND_VALIDATION:", RUN_STATISTICS_AND_VALIDATION)
print("RUN_ASR_EXPERIMENTS:", RUN_ASR_EXPERIMENTS)
print("RUN_LLM_CLINICAL_MARKERS:", RUN_LLM_CLINICAL_MARKERS)
print("RUN_LLM_SCENE_GRAPH_BATCHES:", RUN_LLM_SCENE_GRAPH_BATCHES)
print("RUN_LLM_EVENT_TRIPLE_BATCHES:", RUN_LLM_EVENT_TRIPLE_BATCHES)
print("RUN_LLM_EXTRA_EXPLORATIONS:", RUN_LLM_EXTRA_EXPLORATIONS)
print("RUN_EXPLORATORY_NO_API_EXPERIMENTS:", RUN_EXPLORATORY_NO_API_EXPERIMENTS)
print("RUN_HUBERT_LAYER_SWEEP:", RUN_HUBERT_LAYER_SWEEP)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

drop_path = Path(DROP_DIR)
project_path = Path(PROJECT_DIR)
data_zip_path = Path(DATA_ZIP)

if not drop_path.exists():
    raise FileNotFoundError(f"DROP_DIR not found: {drop_path}")
if not project_path.exists():
    raise FileNotFoundError(f"PROJECT_DIR not found: {project_path}")
for required in [project_path / "scripts", project_path / "src", project_path / "requirements.txt"]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required project item: {required}")
if not data_zip_path.exists():
    raise FileNotFoundError(f"DATA_ZIP not found. Edit DATA_ZIP: {data_zip_path}")

llm_requested = any([
    RUN_LLM_CLINICAL_MARKERS,
    RUN_LLM_SCENE_GRAPH_BATCHES,
    RUN_LLM_EVENT_TRIPLE_BATCHES,
    RUN_LLM_EXTRA_EXPLORATIONS,
])
if llm_requested and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("An LLM section was requested, but OPENAI_API_KEY is not set.")

print("Drive/package validation complete.")

## Dataset Extraction And Environment

The dataset is extracted to local Colab storage so WAV reads do not stream from Drive. The project package itself should not contain raw data.

In [ ]:
extract_path = Path(EXTRACT_DIR)
if extract_path.exists():
    shutil.rmtree(extract_path)
extract_path.mkdir(parents=True, exist_ok=True)

print("Extracting", DATA_ZIP, "to", EXTRACT_DIR)
with zipfile.ZipFile(DATA_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

def find_dataset_dir(root: Path) -> Path:
    candidates = []
    for path in [root, *root.rglob("*")]:
        if path.is_dir() and all((path / name).exists() for name in ["splits", "sessions", "wavs"]):
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError("Could not find extracted dataset folder containing splits/, sessions/, wavs/.")
    return sorted(candidates, key=lambda p: len(p.parts))[0]

DATA_DIR = str(find_dataset_dir(extract_path))
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("DATA_DIR:", DATA_DIR)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print("Python:", sys.version)
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: no GPU detected. Transformer/audio embedding experiments may be slow.")
except Exception as exc:
    print("torch import failed:", repr(exc))

In [ ]:
def run_commands(commands, enabled=True, title="commands"):
    if not enabled:
        print(f"Skipping {title}")
        return
    for command in commands:
        print("+", " ".join(str(part) for part in command))
        subprocess.run(command, check=True)

PY = sys.executable

## Required Pipeline

These commands reproduce the core required experiments and the final defensible model inputs. The transformer baselines may take substantial GPU time on a fresh Colab runtime.

In [ ]:
required_commands = [
    [PY, "scripts/audit_dataset.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_text_baseline.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_descriptor_baselines.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_acoustic_descriptor_baseline.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_roberta_text_baseline.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_hubert_baseline.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_wavlm_baseline.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_fusion.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_clinical_thresholding.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_linguistic_marker_panel.py", "--data_dir", DATA_DIR],
    [PY, "scripts/analyze_marker_errors.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_uncertainty_gated_marker_fusion.py"],
    [PY, "scripts/run_marker_error_corrector.py"],
    [PY, "scripts/run_stacked_tfidf_marker_meta_model.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_marker_stratified_thresholding.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_age_aware_thresholding.py", "--data_dir", DATA_DIR],
]
run_commands(required_commands, RUN_REQUIRED_PIPELINE, "required pipeline")

## Final Analyses Used In The Report

These commands regenerate or summarize the final scene-fusion winner, calibrated operating points, statistical comparisons, and metadata/MMSE validation used in the written summary.

In [ ]:
final_analysis_commands = [
    [PY, "scripts/analyze_scene_graph_winner.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_final_calibrated_scene_fusion.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_statistical_significance_official_baseline.py"],
    [PY, "scripts/run_statistical_significance_tests.py"],
    [PY, "scripts/run_metadata_clinical_validation.py", "--data_dir", DATA_DIR],
]
run_commands(final_analysis_commands, RUN_FINAL_ANALYSES and RUN_STATISTICS_AND_VALIDATION, "final analyses")

## Optional ASR Experiments Tried

These experiments use Whisper/ASR outputs and are slower than the core text/audio baselines. They were exploratory and did not replace the final scene-fusion model.

In [ ]:
asr_commands = [
    [PY, "scripts/run_asr_error_features.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/asr_error_features", "--whisper_model", "base"],
    [PY, "scripts/run_asr_error_profile_fusion.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_asr_transcript_text_fusion.py", "--data_dir", DATA_DIR],
]
run_commands(asr_commands, RUN_ASR_EXPERIMENTS, "ASR experiments")

## Optional LLM Clinical Marker Experiments Tried

These use transcript text and anonymous session IDs only. They require `OPENAI_API_KEY` and should be rerun only when API cost is acceptable.

In [ ]:
llm_clinical_commands = [
    [PY, "scripts/run_llm_clinical_marker_panel.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/llm_clinical_marker_panel", "--model", LLM_MODEL],
    [PY, "scripts/run_llm_stratified_thresholding.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_llm_fusion_optimization.py", "--data_dir", DATA_DIR],
]
run_commands(llm_clinical_commands, RUN_LLM_CLINICAL_MARKERS, "LLM clinical marker experiments")

## Optional Cookie Theft Scene-Graph Batches

The final credible model uses cached Cookie Theft scene/discourse features. This section is manual and resumable. Run the progress cell, then repeatedly run the batch cell until missing sessions are zero, then run the final modeling cell.

In [ ]:
if RUN_LLM_SCENE_GRAPH_BATCHES:
    run_commands([[
        PY, "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_scene_graph",
        "--model", LLM_MODEL,
        "--list_progress",
    ]], True, "scene-graph progress")
else:
    print("Skipping scene-graph API progress check")

In [ ]:
# Rerun this cell until the progress check reports zero missing sessions.
if RUN_LLM_SCENE_GRAPH_BATCHES:
    run_commands([[
        PY, "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_scene_graph",
        "--model", LLM_MODEL,
        "--only_missing",
        "--batch_size", str(LLM_SCENE_GRAPH_BATCH_SIZE),
    ]], True, "scene-graph next missing batch")
else:
    print("Skipping scene-graph API batch")

In [ ]:
if RUN_LLM_SCENE_GRAPH_BATCHES:
    run_commands([[
        PY, "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_scene_graph",
        "--model", LLM_MODEL,
        "--only_missing",
    ]], True, "scene-graph cached modeling")
else:
    print("Skipping scene-graph cached modeling")

## Optional Event-Triple Batches Tried

Event triples were explored as another task-specific representation. They are retained for traceability but are not the final headline model.

In [ ]:
if RUN_LLM_EVENT_TRIPLE_BATCHES:
    run_commands([[
        PY, "scripts/run_llm_cookie_theft_event_triples.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_event_triples",
        "--model", LLM_MODEL,
        "--list_progress",
    ]], True, "event-triple progress")
else:
    print("Skipping event-triple progress check")

In [ ]:
# Rerun this cell until the progress check reports zero missing sessions.
if RUN_LLM_EVENT_TRIPLE_BATCHES:
    run_commands([[
        PY, "scripts/run_llm_cookie_theft_event_triples.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_event_triples",
        "--model", LLM_MODEL,
        "--only_missing",
        "--batch_size", str(LLM_EVENT_TRIPLE_BATCH_SIZE),
    ]], True, "event-triple next missing batch")
else:
    print("Skipping event-triple API batch")

In [ ]:
if RUN_LLM_EVENT_TRIPLE_BATCHES:
    run_commands([[
        PY, "scripts/run_llm_cookie_theft_event_triples.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_event_triples",
        "--model", LLM_MODEL,
        "--only_missing",
    ]], True, "event-triple cached modeling")
else:
    print("Skipping event-triple cached modeling")

## Optional Extra LLM Explorations Tried

These are exploratory LLM feature and adjudication variants. They are included for completeness and are not used as the final headline result.

In [ ]:
llm_extra_commands = [
    [PY, "scripts/run_event_triples_auto_batches.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_llm_hard_case_adjudicator.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_hard_case_auto_batches.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_llm_multiview_scene_profile.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_multiview_scene_profile_auto_batches.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_llm_multiview_evidence_profile.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_multiview_evidence_profile_auto_batches.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_llm_scene_view_aggregation.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_llm_scene_view_aggregation_auto_batches.py", "--data_dir", DATA_DIR],
]
run_commands(llm_extra_commands, RUN_LLM_EXTRA_EXPLORATIONS, "extra LLM explorations")

## Optional No-API Exploratory Experiments Tried

These experiments test alternative decision structures, ablations, triage strategies, semantic/disfluency features, and subject-level aggregation. They document model search and negative results but are not all comparable to the session-level final model.

In [ ]:
exploratory_no_api_commands = [
    [PY, "scripts/run_hubert_layer_sweep.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_scene_graph_fusion_optimization.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_scene_graph_feature_subset_search.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_final_decision_structure_experiments.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_disagreement_aware_scene_fusion.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_cascade_triage_experiment.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_transcript_view_aggregation.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_pause_disfluency_fusion.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_two_pass_uncertainty_resolver.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_tfidf_plus_scene_ablation.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_semantic_specificity_fusion.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_subject_feature_aggregation.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_session_aggregation_ablation.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_final_push_no_api.py", "--data_dir", DATA_DIR],
    [PY, "scripts/run_literature_inspired_final_push.py", "--data_dir", DATA_DIR],
]
run_commands(exploratory_no_api_commands, RUN_EXPLORATORY_NO_API_EXPERIMENTS or RUN_HUBERT_LAYER_SWEEP, "no-API exploratory experiments")

## Experiment Inventory

This cell records the full experiment surface represented in the repository. Status values distinguish default rerun sections from optional/API/manual explorations and saved-summary-only artifacts.

In [ ]:
import pandas as pd
from IPython.display import display

inventory_rows = [
    ("Dataset audit", "scripts/audit_dataset.py", "default-run", "Split/session/WAV sanity checks"),
    ("TF-IDF text baseline", "scripts/run_text_baseline.py", "default-run", "Official strong text baseline"),
    ("Descriptor baselines", "scripts/run_descriptor_baselines.py", "default-run", "Age/sex/speech quantity baselines"),
    ("Lightweight acoustic descriptors", "scripts/run_acoustic_descriptor_baseline.py", "default-run", "Non-transformer acoustic baseline"),
    ("Frozen RoBERTa text", "scripts/run_roberta_text_baseline.py", "default-run", "Transformer text embedding baseline"),
    ("HuBERT final layer", "scripts/run_hubert_baseline.py", "default-run", "Official acoustic embedding baseline"),
    ("WavLM final layer", "scripts/run_wavlm_baseline.py", "default-run", "Alternative acoustic embedding baseline"),
    ("TF-IDF + HuBERT fusion", "scripts/run_fusion.py", "default-run", "Weighted late fusion"),
    ("Clinical thresholding", "scripts/run_clinical_thresholding.py", "default-run", "Recall-constrained thresholds"),
    ("Linguistic marker panel", "scripts/run_linguistic_marker_panel.py", "default-run", "Interpretable marker features"),
    ("Marker error analysis", "scripts/analyze_marker_errors.py", "default-run", "Error groups and marker interpretation"),
    ("Uncertainty-gated marker fusion", "scripts/run_uncertainty_gated_marker_fusion.py", "default-run", "Near-boundary marker fusion"),
    ("Marker error corrector", "scripts/run_marker_error_corrector.py", "default-run", "Correction-grid exploration"),
    ("Stacked TF-IDF marker meta-model", "scripts/run_stacked_tfidf_marker_meta_model.py", "default-run", "OOF marker/text stack"),
    ("Marker stratified thresholding", "scripts/run_marker_stratified_thresholding.py", "default-run", "Marker-conditioned thresholds"),
    ("Age-aware thresholding", "scripts/run_age_aware_thresholding.py", "default-run", "Age-stratified threshold analysis"),
    ("Scene graph winner analysis", "scripts/analyze_scene_graph_winner.py", "default-run final", "Final headline model analysis"),
    ("Final calibrated scene fusion", "scripts/run_final_calibrated_scene_fusion.py", "default-run final", "Final operating-point analysis"),
    ("Official baseline significance", "scripts/run_statistical_significance_official_baseline.py", "default-run final", "TF-IDF vs scene winner tests"),
    ("Fusion significance tests", "scripts/run_statistical_significance_tests.py", "default-run final", "Scene winner vs other baselines"),
    ("Metadata/MMSE validation", "scripts/run_metadata_clinical_validation.py", "default-run final", "Exploratory clinical validation"),
    ("ASR error features", "scripts/run_asr_error_features.py", "optional slow", "Whisper-derived ASR error features"),
    ("ASR error profile fusion", "scripts/run_asr_error_profile_fusion.py", "optional slow", "ASR-profile fusion/error analysis"),
    ("ASR transcript text fusion", "scripts/run_asr_transcript_text_fusion.py", "optional slow", "ASR transcript TF-IDF fusion"),
    ("LLM clinical marker panel", "scripts/run_llm_clinical_marker_panel.py", "optional API", "Observable language marker scoring"),
    ("LLM stratified thresholding", "scripts/run_llm_stratified_thresholding.py", "optional API", "LLM-feature thresholding"),
    ("LLM fusion optimization", "scripts/run_llm_fusion_optimization.py", "optional API", "LLM feature fusion grids"),
    ("Cookie Theft scene graph", "scripts/run_llm_cookie_theft_scene_graph.py", "optional/manual API", "Task-grounded scene/discourse features"),
    ("Cookie Theft event triples", "scripts/run_llm_cookie_theft_event_triples.py", "optional/manual API", "Task event triple features"),
    ("Event triple auto batches", "scripts/run_event_triples_auto_batches.py", "optional API", "Batch helper for event triples"),
    ("LLM hard-case adjudicator", "scripts/run_llm_hard_case_adjudicator.py", "optional API", "Hard-case reranker/adjudication"),
    ("Hard-case auto batches", "scripts/run_hard_case_auto_batches.py", "optional API", "Batch helper for hard cases"),
    ("LLM multiview scene profile", "scripts/run_llm_multiview_scene_profile.py", "optional API", "Multiview scene profile"),
    ("LLM multiview evidence profile", "scripts/run_llm_multiview_evidence_profile.py", "optional API", "Multiview evidence profile"),
    ("LLM scene view aggregation", "scripts/run_llm_scene_view_aggregation.py", "optional API", "Generated scene views"),
    ("HuBERT layer sweep", "scripts/run_hubert_layer_sweep.py", "optional slow", "Layers 0-12 exploration"),
    ("Scene graph fusion optimization", "scripts/run_scene_graph_fusion_optimization.py", "optional exploration", "Fusion and threshold grids"),
    ("Scene graph subset search", "scripts/run_scene_graph_feature_subset_search.py", "optional exploration", "Feature subset search"),
    ("Final decision structures", "scripts/run_final_decision_structure_experiments.py", "optional exploration", "Subject/triage decision alternatives"),
    ("Disagreement-aware scene fusion", "scripts/run_disagreement_aware_scene_fusion.py", "optional exploration", "Disagreement/correction rules"),
    ("Cascade triage", "scripts/run_cascade_triage_experiment.py", "optional exploration", "Two-stage cascade/abstention"),
    ("Transcript view aggregation", "scripts/run_transcript_view_aggregation.py", "optional exploration", "Transcript view augmentation"),
    ("Pause/disfluency fusion", "scripts/run_pause_disfluency_fusion.py", "optional exploration", "Disfluency features"),
    ("Two-pass uncertainty resolver", "scripts/run_two_pass_uncertainty_resolver.py", "optional exploration", "Uncertainty routing"),
    ("TF-IDF plus scene ablation", "scripts/run_tfidf_plus_scene_ablation.py", "optional exploration", "Scene/text ablation"),
    ("Semantic specificity fusion", "scripts/run_semantic_specificity_fusion.py", "optional exploration", "Semantic specificity features"),
    ("Subject feature aggregation", "scripts/run_subject_feature_aggregation.py", "optional exploration", "Subject-level aggregation"),
    ("Session aggregation ablation", "scripts/run_session_aggregation_ablation.py", "optional exploration", "Session aggregation variants"),
    ("Final no-API push", "scripts/run_final_push_no_api.py", "optional exploration", "No-API correction/ensemble variants"),
    ("Literature-inspired final push", "scripts/run_literature_inspired_final_push.py", "optional exploration", "Prototype/profile experiments"),
]

inventory = pd.DataFrame(inventory_rows, columns=["experiment", "script", "status", "purpose"])
inventory["script_present"] = inventory["script"].map(lambda p: Path(p).exists())
inventory["result_md_present"] = inventory["script"].map(lambda p: False)
result_dirs = {p.parent.name for p in Path("outputs").glob("*/results.md")}
inventory["saved_result_group"] = inventory["script"].str.replace("scripts/run_", "", regex=False).str.replace("scripts/analyze_", "", regex=False).str.replace(".py", "", regex=False)
inventory["saved_result_present"] = inventory["saved_result_group"].map(lambda g: g in result_dirs)
display(inventory)

missing_scripts = inventory.loc[~inventory["script_present"], "script"].tolist()
if missing_scripts:
    raise FileNotFoundError(f"Missing scripts listed in inventory: {missing_scripts}")

## Result Summary Display

These cells print the compact saved summaries that should be used in the report. They avoid exposing raw transcripts or raw response caches.

In [ ]:
subprocess.run([PY, "scripts/collect_required_results.py"], check=True)

summary_paths = [
    "outputs/required_results_summary/summary.md",
    "outputs/required_results_summary/results_table.md",
    "reports/result_summary.md",
    "outputs/reports/final_results_summary_source.md",
    "outputs/scene_graph_winner_analysis/results.md",
    "outputs/final_calibrated_scene_fusion/results.md",
    "outputs/statistical_significance_official_baseline/results.md",
    "outputs/statistical_significance_tests/results.md",
    "outputs/metadata_clinical_validation/results.md",
]
for rel in summary_paths:
    path = Path(rel)
    print("\n" + "=" * 100)
    print(rel)
    print("=" * 100)
    if path.exists():
        print(path.read_text()[:12000])
    else:
        print("Missing:", rel)

## Submission Packaging

This creates two zip files in the Drive package output folder: a full generated-output archive and a compact summary archive. The summary archive is usually the safer item to submit alongside the notebook/report because it excludes raw data and caches.

In [ ]:
subprocess.run([PY, "scripts/collect_required_results.py"], check=True)

output_zip = Path(OUTPUT_ZIP)
summary_zip = Path(SUMMARY_ZIP)
output_zip.parent.mkdir(parents=True, exist_ok=True)
for path in [output_zip, summary_zip]:
    if path.exists():
        path.unlink()

# Full generated outputs. This is useful for traceability, but inspect size/content before submitting.
shutil.make_archive(str(output_zip.with_suffix("")), "zip", root_dir=PROJECT_DIR, base_dir="outputs")

summary_files = [
    "outputs/required_results_summary/results_table.md",
    "outputs/required_results_summary/results_table.csv",
    "outputs/required_results_summary/summary.md",
    "outputs/tables/required_results_table.md",
    "outputs/tables/required_results_table.csv",
    "outputs/tables/best_results_table.csv",
    "reports/result_summary.md",
    "outputs/reports/final_results_summary_source.md",
    "outputs/text_baseline/results.md",
    "outputs/acoustic_descriptor_baseline/results.md",
    "outputs/descriptor_baselines/results.md",
    "outputs/roberta_text_baseline/results.md",
    "outputs/hubert_baseline/results.md",
    "outputs/wavlm_baseline/results.md",
    "outputs/fusion/results.md",
    "outputs/clinical_thresholding/results.md",
    "outputs/linguistic_marker_panel/results.md",
    "outputs/linguistic_marker_panel/error_analysis/marker_error_analysis_summary.md",
    "outputs/uncertainty_gated_marker_fusion/results.md",
    "outputs/marker_error_corrector/results.md",
    "outputs/stacked_tfidf_marker_meta_model/results.md",
    "outputs/marker_stratified_thresholding/results.md",
    "outputs/age_aware_thresholding/results.md",
    "outputs/scene_graph_winner_analysis/results.md",
    "outputs/final_calibrated_scene_fusion/results.md",
    "outputs/statistical_significance_official_baseline/results.md",
    "outputs/statistical_significance_tests/results.md",
    "outputs/metadata_clinical_validation/results.md",
    "outputs/hubert_layer_sweep/results.md",
    "outputs/asr_error_features/results.md",
    "outputs/asr_error_profile_fusion/results.md",
    "outputs/asr_transcript_text_fusion/results.md",
    "outputs/llm_clinical_marker_panel/results.md",
    "outputs/llm_stratified_thresholding/results.md",
    "outputs/llm_fusion_optimization/results.md",
    "outputs/llm_cookie_theft_scene_graph/results.md",
    "outputs/llm_cookie_theft_event_triples/results.md",
    "outputs/llm_hard_case_adjudicator/results.md",
    "outputs/scene_graph_fusion_optimization/results.md",
    "outputs/scene_graph_feature_subset_search/results.md",
    "outputs/final_decision_structure_experiments/results.md",
    "outputs/disagreement_aware_scene_fusion/results.md",
    "outputs/cascade_triage_experiment/results.md",
    "outputs/transcript_view_aggregation/results.md",
    "outputs/pause_disfluency_fusion/results.md",
    "outputs/two_pass_uncertainty_resolver/results.md",
    "outputs/tfidf_plus_scene_ablation/results.md",
    "outputs/semantic_specificity_fusion/results.md",
    "outputs/subject_feature_aggregation/results.md",
    "outputs/final_push_no_api/results.md",
    "outputs/literature_inspired_final_push/results.md",
    "outputs/llm_multiview_scene_profile/results.md",
    "outputs/llm_multiview_evidence_profile/results.md",
    "outputs/llm_scene_view_aggregation/results.md",
]

with zipfile.ZipFile(summary_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in summary_files:
        path = Path(PROJECT_DIR) / rel
        if path.exists():
            zf.write(path, rel)

print("Full output zip:", output_zip)
print("Full output zip size MB:", round(output_zip.stat().st_size / (1024 * 1024), 2))
print("Summary zip:", summary_zip)
print("Summary zip size KB:", round(summary_zip.stat().st_size / 1024, 1))

## Final Submission Notes

The final credible session-level headline model is TF-IDF + HuBERT + Cookie Theft scene/discourse fusion from `outputs/scene_graph_winner_analysis/results.json`. The saved test metrics are AUROC 0.9257, F1 0.8537, accuracy 0.8788, precision 0.8974, and recall 0.8140.

Exploratory models with different decision units, abstention, repeated test-observed search, or API-heavy reranking are reported as exploratory rather than replacing the final forced session-level classifier.